# cme quickstart: does a multi-agent system beat the self-consistency frontier?

This notebook runs the `cme` tools on **synthetic demo data** (`code/cme/examples/make_demo.py`, seed 7). The numbers come from fixed random rules, not from any model, and say nothing about any real method.

Run it from the repository root (upload or clone the repository; no public URL is assumed). It needs Python 3.9+, `numpy` and `matplotlib`.

In [ ]:
# Cell 1: install the packages from the local repository (cme and mae_eval)
!pip install -q -e code

In [ ]:
# Cell 2: quickstart evaluation on the bundled synthetic pool and run log
import os, subprocess, sys
os.environ.setdefault("PYTHONPATH", "code")
OUT = os.environ.get("CME_NB_OUT", "cme_demo_out")
os.makedirs(OUT, exist_ok=True)
POOL, RUN = "code/cme/examples/demo_pool.jsonl", "code/cme/examples/demo_run.jsonl"

def cme(*args, module="cme"):
    r = subprocess.run([sys.executable, "-m", module, *args], capture_output=True, text=True)
    print(r.stdout + r.stderr)
    return r

# frontier gap with a paired bootstrap interval, resource ratios (n/a when both sides are 0), LaTeX card
cme("report", "--pool", POOL, "--logs", RUN, "--system", "debate", "--n-boot", "500",
    "--latex", OUT + "/report_card.tex", "--json", OUT + "/report.json")
# the mae_eval auditor on a hand-written synthetic trace (single agent versus three workers plus a judge)
cme("audit", "--log", "code/examples/sample_trace.jsonl", "--price-in", "1.0", "--price-out", "4.0", module="mae_eval")

In [ ]:
# Cell 3: the self-consistency frontier with the multi-agent systems of the run log (Okabe-Ito palette)
cme("frontier", "--samples", POOL, "--logs", RUN, "--out", OUT + "/frontier.png")
cme("frontier", "--samples", POOL, "--logs", RUN, "--out", OUT + "/frontier.pdf")   # vector version for a paper
try:
    from IPython.display import Image, display
    display(Image(OUT + "/frontier.png"))
except Exception:
    print("figure written to", OUT + "/frontier.png")

In [ ]:
# Cell 4: the nine-column rebuttal table as LaTeX, from the bundled literature contrasts
cme("rebuttal", "--data", "outputs/baseline_tier_contrasts.csv", "--study", "Tran", "--export-latex", OUT + "/rebuttal_table.tex")
print(open(OUT + "/rebuttal_table.tex").read()[:900])

In [ ]:
# Cell 5: `cme audit`, `cme hidden-profile` and `cme dynamics` on synthetic data
import json, random
cme("audit", "code/examples/sample_trace.jsonl")                                   # one-screen card: rung, MAS/SAS ratios, badge, BibTeX
cme("hidden-profile", "--n", "5", "--agents", "5", "--criteria", "10", "--split", "6", "--margin", "3", "--seed", "42", "--out", OUT + "/hidden_profile.jsonl")
# a synthetic debate log with known transition rates (wrong->right 0.30, right->wrong 0.10): the command should recover them approximately
rng = random.Random(3)
with open(OUT + "/debate_log.jsonl", "w") as fh:
    for i in range(200):
        state = [rng.random() < 0.7 for _ in range(5)]
        for rnd in (1, 2):
            for a, s in enumerate(state):
                fh.write(json.dumps({"item": "q%d" % i, "agent": a, "round": rnd, "correct": s, "prompt_tokens": 100 * rnd, "completion_tokens": 50}) + "\n")
            state = [(rng.random() >= 0.10) if s else (rng.random() < 0.30) for s in state]
cme("dynamics", OUT + "/debate_log.jsonl", "--n-boot", "200", "--seed", "0")